In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [21]:
print(data["Precio Spot"].loc["2025-02-24 00:01:00.692"])

1.047825


In [2]:
import find_best
import keys
import pandas as pd
from use_tecnics import simple_methods

keys.candles = 20
keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=3) + pd.Timedelta(days=4)
inicio_testeo = pd.Timedelta(days=3)
fin_testeo = pd.Timedelta(days=4)

inicio_train = data.index[0].normalize()
fin_datos = data.index[-1]

while True:
    train_time = inicio_train + ventana_entrenamiento
    inicio_test = train_time + inicio_testeo
    fin_test = inicio_test + fin_testeo
    print(f"Entrenamiento: {inicio_train.strftime('%Y-%m-%d')} a {train_time.strftime('%Y-%m-%d')}")

    sub_data = data[inicio_train: train_time]
    best_ma = find_best.opti_main(sub_data)

    inicio_train = inicio_train + pd.Timedelta(days=7)
    if train_time >= fin_datos:
        break

    minutos_de_colchon = best_ma[1]*best_ma[2]*(1 if best_ma[0] in simple_methods else 8)
    minutos_de_colchon = train_time - pd.Timedelta(minutes=minutos_de_colchon)
    sub_data = pd.concat([sub_data[minutos_de_colchon:],
                          data[inicio_test: fin_test]])

    print(f"\nTesteo: {inicio_test.strftime('%Y-%m-%d')} a {fin_test.strftime('%Y-%m-%d')}")
    find_best.read_results(best_ma, sub_data)

Entrenamiento: 2025-02-24 a 2025-03-21
Resultado obtenido entrenando desde 2025-02-24 hasta 2025-03-20
Método: T3, Datos optimizados [np.int64(16), np.int64(108)]

hit ratio: 0.5319148936170213
risk reward: 1.6476462196862767
profit factor: 1.8723252496434961
trades: 47
Resultado de sobre ajuste -1.3546642711379933

Testeo: 2025-03-24 a 2025-03-28
Rendimiento de una ma con:
método: T3, vela: 16, añadidos: [np.int64(108)]
hit ratio: 0.5555555555555556
risk reward: 3.4000000000007122
profit factor: 4.25000000000089
trades: 9


Entrenamiento: 2025-03-03 a 2025-03-28
Resultado obtenido entrenando desde 2025-03-03 hasta 2025-03-27
Método: EMA, Datos optimizados [np.int64(17), np.int64(86)]

hit ratio: 0.6309523809523809
risk reward: 0.9572846236445685
profit factor: 1.6366479049407137
trades: 84
Resultado de sobre ajuste -1.51220615955591

Testeo: 2025-03-31 a 2025-04-04
Rendimiento de una ma con:
método: EMA, vela: 17, añadidos: [np.int64(86)]
hit ratio: 0.5833333333333334
risk reward: 1.6